In [1]:
from rouge_score import rouge_scorer
from bert_score import score as bertscore

c:\Users\BLACKBOX\.anaconda-desktop\micromamba\envs\cuda\envs\condaenv1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def evaluate_summary(generated_summary, reference_summary):

    if not generated_summary or not reference_summary:
        return {
            "ROUGE1": 0,
            "ROUGE2": 0,
            "ROUGEL": 0,
            "BERTScore": 0
        }

    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'],
        use_stemmer=True
    )

    scores = scorer.score(reference_summary, generated_summary)

    P, R, F1 = bertscore(
        [generated_summary],
        [reference_summary],
        lang="en",
        verbose=False
    )

    return {
        "ROUGE1": scores['rouge1'].fmeasure,
        "ROUGE2": scores['rouge2'].fmeasure,
        "ROUGEL": scores['rougeL'].fmeasure,
        "BERTScore": F1.mean().item()
    }

In [3]:
def run_test(
    pipeline_name,
    model_name,
    input_text,
    reference_summary,
    paper_id
):
    truncated_text = " ".join(
        str(input_text).split()[:800]
    )

    process = psutil.Process()

    start_mem = process.memory_info().rss / (1024 * 1024)
    start_time = time.time()

    try:

        response = ollama.chat(
            model=model_name,
            messages=[
                {
                    'role': 'user',
                    'content':
                    f"Summarize this scientific text in two sentences:\n\n{truncated_text}"
                }
            ]
        )

        output_text = response['message']['content']

    except Exception as e:

        output_text = f"Error: {str(e)}"

    end_time = time.time()
    end_mem = process.memory_info().rss / (1024 * 1024)

    latency = end_time - start_time

    memory_used = max(
        0,
        end_mem - start_mem
    )

    metrics = evaluate_summary(
        output_text,
        reference_summary
    )

    return {

        "Paper_ID": paper_id,

        "Pipeline": pipeline_name,

        "Latency_Sec": round(latency,3),

        "RAM_Used_MB": round(memory_used,2),

        "Output_Word_Count": len(output_text.split()),

        "Compression_Ratio":
            len(output_text.split()) /
            max(1, len(truncated_text.split())),

        "ROUGE1": round(metrics["ROUGE1"],4),

        "ROUGE2": round(metrics["ROUGE2"],4),

        "ROUGEL": round(metrics["ROUGEL"],4),

        "BERTScore": round(metrics["BERTScore"],4),

        "Summary": output_text
    }